<a href="https://colab.research.google.com/github/VijayaRagav07/MMB/blob/main/InitialMMB_Not_Optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)



Mounted at /content/drive


In [ ]:
import os
BASE_PATH = "/content/drive/MyDrive/MMB datsets"



MODALITIES = {
    "ultrasound": "Ultrasound Images_MSI",
    "histopath": "Histopathological_MSI",
    "xray": "Chest_XRay_MSI"
}

In [ ]:

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

In [ ]:
def load_images(path):
    images = []
    labels = []

    class_names = ["Benign", "Malignant"]  # FIXED (capital letters)

    for label, class_name in enumerate(class_names):
        class_path = os.path.join(path, class_name)

        if not os.path.exists(class_path):
            print("❌ Missing path:", class_path)
            continue

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)
            if img is None:
                continue

            img = cv2.resize(img, (224, 224))

            # RGB
            rgb = img

            # HSV
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            # JET
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            jet = cv2.applyColorMap(gray, cv2.COLORMAP_JET)

            rgb = rgb / 255.0
            hsv = hsv / 255.0
            jet = jet / 255.0

            combined = np.concatenate([rgb, hsv, jet], axis=-1)

            images.append(combined)
            labels.append(label)

    return np.array(images), np.array(labels)

In [ ]:
print("\nLoading ultrasound...")
X_ultrasound, y_ultrasound = load_images(os.path.join(BASE_PATH, MODALITIES["ultrasound"]))
print("Ultrasound shape:", X_ultrasound.shape)

print("\nLoading histopath...")
X_histopath, y_histopath = load_images(os.path.join(BASE_PATH, MODALITIES["histopath"]))
print("Histopath shape:", X_histopath.shape)

print("\nLoading xray...")
X_xray, y_xray = load_images(os.path.join(BASE_PATH, MODALITIES["xray"]))
print("X-ray shape:", X_xray.shape)

print("\n✅ All data loaded successfully!")


Loading ultrasound...
Ultrasound shape: (806, 224, 224, 9)

Loading histopath...


In [ ]:
def build_model():
    inputs = layers.Input(shape=(224,224,9))

    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
def train_on_modality(modality):
    X, y = data[modality]

    # Shuffle
    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]

    split = int(0.8 * len(X))
    X_train, X_val = X[:split], X[split:]
    y_train, y_val = y[:split], y[split:]

    model = build_model()

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=16
    )

    return model

In [ ]:

def evaluate_model(model, X, y, name):
    preds = model.predict(X)
    preds_binary = (preds > 0.5).astype(int)

    print(f"\n===== Evaluation on {name} =====")

    print("Classification Report:")
    print(classification_report(y, preds_binary))

    print("Confusion Matrix:")
    print(confusion_matrix(y, preds_binary))

    try:
        auc = roc_auc_score(y, preds)
        print("ROC-AUC:", auc)
    except:
        print("ROC-AUC not available")

In [ ]:
# Train on Ultrasound
model = train_on_modality("ultrasound")

# Test on ALL modalities
for key in data:
    X_test, y_test = data[key]
    evaluate_model(model, X_test, y_test, key)

In [ ]:
model = train_on_modality("histopath")

# Test on ALL modalities
for key in data:
    X_test, y_test = data[key]
    evaluate_model(model, X_test, y_test, key)

In [ ]:
model = train_on_modality("xray")

# Test on ALL modalities
for key in data:
    X_test, y_test = data[key]
    evaluate_model(model, X_test, y_test, key)